# Training a DFlash Draft Model for Cosmos3 Nano

This notebook walks through online DFlash training with a Cosmos3 Nano vision-language model as the frozen target. DFlash learns a compact block-diffusion draft model; unlike EAGLE3, it does not use a calibrated draft vocabulary.

| Step | Description |
| :---: | :--- |
| 1 | Install dependencies from this checkout |
| 2 | Optionally authenticate with Hugging Face |
| 3 | Prepare multimodal training data *(deferred)* |
| 4 | Launch online DFlash training |
| 5 | Export the DFlash checkpoint for deployment |

> **Hardware requirement** – the example launches eight local GPU processes. Set `NUM_GPUS` lower for a smoke test only if the Cosmos3 Nano target fits in the available GPU memory.


## Step 1 – Install Dependencies

Run this notebook from `examples/speculative_decoding/recipes`. The editable install ensures that training uses this checkout's DFlash and VLM support.


In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
python3 -m pip install -r "$REPO_ROOT/examples/speculative_decoding/requirements.txt"
python3 -m pip install -e "$REPO_ROOT[hf]"


## Step 2 – Authenticate with Hugging Face (Optional)

This is needed only when the Cosmos3 Nano checkpoint, processor, or training data is downloaded from the Hub. It is not needed for already-local paths.


In [ ]:
%%bash
hf auth login


## Step 3 – Prepare Multimodal Training Data

This step is intentionally deferred. The training JSONL must contain OpenAI-style `messages`, including image and/or video content parts, and the media paths must resolve below `data.vlm_img_dir`.

For DFlash, choose a `training_seq_len` divisible by `dflash_block_size` (the example below uses `16384` and `8`). Keep the model's ChatML template or provide a compatible assistant-mask template when using `answer_only_loss=true`.

> Add the data-preparation workflow here once the final Cosmos3 Nano multimodal dataset and media layout are settled.


## Step 4 – Train the DFlash Draft Model

DFlash uses the online speculative-decoding recipe directly. It loads Cosmos3 Nano as the target, collects its hidden states through the top-level VLM forward, and trains a five-layer Qwen3-style draft decoder.

Update the four paths below before running the cell. `TRAINING_DATA` is the JSONL produced by Step 3. The architecture settings are pinned to Cosmos3 Nano's text tower rather than inferred from generic Qwen3 defaults.


In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"

# Replace these paths with the local Cosmos3 Nano checkpoint and Step 3 data.
MODEL_PATH=/path/to/cosmos3-nano-reasoner
TRAINING_DATA=/path/to/cosmos3-nano-train.jsonl
VLM_IMG_DIR=/
OUTPUT_DIR="$REPO_ROOT/ckpts/cosmos3-nano-dflash"
NUM_GPUS=${NUM_GPUS:-8}

for path in "$MODEL_PATH" "$TRAINING_DATA"; do
  case "$path" in
    /path/to/*) echo "Set MODEL_PATH and TRAINING_DATA before training." >&2; exit 1 ;;
  esac
done

export WANDB_MODE=disabled
export TOKENIZERS_PARALLELISM=false

cd "$REPO_ROOT"
torchrun --nproc_per_node="$NUM_GPUS" \
  examples/speculative_decoding/main.py \
  --config modelopt_recipes/general/speculative_decoding/dflash.yaml \
  model.model_name_or_path="$MODEL_PATH" \
  model.trust_remote_code=true \
  data.data_path="$TRAINING_DATA" \
  data.vlm_processor="$MODEL_PATH" \
  data.vlm_img_dir="$VLM_IMG_DIR" \
  training.output_dir="$OUTPUT_DIR" \
  training.num_train_epochs=25 \
  training.per_device_train_batch_size=1 \
  training.gradient_accumulation_steps=2 \
  training.training_seq_len=16384 \
  training.answer_only_loss=true \
  training.save_steps=500 \
  training.logging_steps=10 \
  training.dataloader_num_workers=2 \
  training.dataloader_prefetch_factor=2 \
  training.ddp_find_unused_parameters=false \
  training.report_to=none \
  dflash.dflash_block_size=8 \
  dflash.dflash_num_anchors=128 \
  dflash.dflash_loss_objective=decay \
  dflash.dflash_loss_decay_factor=4 \
  dflash.dflash_architecture_config.num_hidden_layers=5 \
  dflash.dflash_architecture_config.num_attention_heads=32 \
  dflash.dflash_architecture_config.num_key_value_heads=8 \
  dflash.dflash_architecture_config.head_dim=128 \
  dflash.dflash_architecture_config.intermediate_size=12288 \
  dflash.dflash_architecture_config.max_position_embeddings=262144 \
  dflash.dflash_architecture_config.rms_norm_eps=1e-06 \
  dflash.dflash_architecture_config.rope_theta=5000000 \
  dflash.dflash_mask_token_id=151669


## Step 5 – Export the DFlash Checkpoint

Export the selected training checkpoint to the DFlash Hugging Face format consumed by vLLM. Replace `checkpoint-<step>` with a saved checkpoint directory.


In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
CKPT_DIR="$REPO_ROOT/ckpts/cosmos3-nano-dflash/checkpoint-<step>"
EXPORT_PATH="$REPO_ROOT/export/cosmos3-nano-dflash"

python3 "$REPO_ROOT/examples/speculative_decoding/scripts/export_hf_checkpoint.py" \
  --model_path "$CKPT_DIR" \
  --export_path "$EXPORT_PATH" \
  --trust_remote_code


## Deployment

Serve the Cosmos3 Nano target with the exported DFlash draft. Keep `num_speculative_tokens` equal to the trained DFlash block size.

```bash
vllm serve /path/to/cosmos3-nano-reasoner \
  --speculative-config '{"method": "dflash", "model": "/path/to/cosmos3-nano-dflash", "num_speculative_tokens": 8}'
```
